# MDS: Multidimensional Scaling

## What is MDS?

**Multidimensional Scaling (MDS)** is a dimensionality reduction technique that, unlike PCA, doesn't work by analyzing feature covariance. Instead, it works directly with **distances (or dissimilarities) between data points**. The goal of MDS is to find a low-dimensional layout of points such that the distances between them, in that new low-dimensional space, match the original pairwise distances as closely as possible.

This makes MDS especially useful when:
- You only have a distance/dissimilarity matrix to begin with (not raw features) — e.g. survey-based similarity scores, or genetic distance between species.
- You want to preserve the **overall geometric relationships** between points (how far apart they are) rather than the directions of maximum variance.
- The data has a curved or nonlinear underlying structure (like a Swiss roll), where straight-line distances in the original space may or may not still be meaningful, depending on the MDS variant used.

## Full Steps to Compute MDS

1. **Compute pairwise distances** between every pair of data points in the original (high-dimensional) space — using a distance metric such as Euclidean distance. This produces an n × n distance matrix, where n is the number of samples.

2. **Define a "stress" function** that measures how well a candidate low-dimensional layout preserves those original distances:

$$
Stress_D(x_1, x_2, \ldots, x_N) = \sqrt{\sum_{i \neq j = 1, \ldots, N} \left( d_{ij} - \lVert x_i - x_j \rVert \right)^2}
$$

   where `dᵢⱼ` is the original distance between points i and j, and `||xᵢ − xⱼ||` is their distance in the new, lower-dimensional coordinates. The closer these two distances are, across all pairs, the lower the stress.

3. **Solve an optimization problem**: find a set of coordinates in the target lower-dimensional space (e.g. 2D) that minimizes this stress — typically via an iterative numerical optimization procedure, since there's no closed-form solution like PCA's eigen-decomposition.

4. **Output the resulting low-dimensional coordinates** — one point per original sample, now embedded in 2 (or however many) dimensions, ready to be plotted or used downstream.

## Key difference from PCA

| | PCA | MDS |
|---|---|---|
| Works on | Feature covariance matrix | Pairwise distances between samples |
| Solution method | Eigen-decomposition (exact, closed-form) | Iterative optimization (minimizing stress) |
| Preserves | Directions of maximum variance | Original pairwise distances |
| Needs raw features? | Yes | No — a distance matrix alone is enough |

In [2]:
# Data manipulation
import pandas as pd
# Import pandas (aliased as pd) for building/manipulating DataFrames —
# likely used to organize the generated points and their labels/colors
# before plotting.

# Visualization
import plotly.express as px
# Import Plotly Express (aliased as px), a high-level plotting library
# for interactive charts. Unlike matplotlib/seaborn used in the PCA
# notebook, Plotly plots are interactive by default (zoomable,
# rotatable) — a natural choice here since MDS is often demonstrated
# on 3D data (like a Swiss roll) that benefits from being able to
# rotate the view to see its shape clearly.

# Skleran
from sklearn.datasets import make_swiss_roll # for creating a swiss roll
# Import make_swiss_roll, a scikit-learn function that generates a
# synthetic 3D dataset shaped like a rolled-up sheet (a "Swiss roll"):
# points that lie on a 2D surface which has been curled into a spiral
# in 3D space. This is a classic toy example for manifold learning
# methods (like MDS, Isomap, LLE, t-SNE) because it's a case where the
# TRUE underlying structure is 2-dimensional, but naive straight-line
# (Euclidean) distances in the raw 3D coordinates badly misrepresent
# how "close" points really are along the rolled surface — a good test
# of whether a method captures the data's real shape.

from sklearn.manifold import MDS # for MDS dimensionality reduction
# Import scikit-learn's MDS class — the actual algorithm implementation
# that will compute a low-dimensional embedding preserving pairwise
# distances, as described before this code arrived.

We create some data using Sklearn’s make_swiss_roll and display it on a 3D plot.

In [5]:
import plotly.io as pio

# pick the one matching where you're running this:
# pio.renderers.default = "jupyterlab"   # JupyterLab
# pio.renderers.default = "vscode"       # VS Code notebooks
pio.renderers.default = "colab"        # Google Colab
# pio.renderers.default = "notebook"     # classic Jupyter Notebook only


# Make a swiss roll
X, y = make_swiss_roll(n_samples=2000, noise=0.05)
# Generate 2000 synthetic 3D points arranged in the "Swiss roll" shape
# described earlier — a 2D surface curled into a spiral in 3D space.
#   - X: shape (2000, 3), the (x, y, z) coordinates of each point.
#   - y: shape (2000,), a continuous value representing each point's
#     position ALONG the rolled-up surface (i.e. how far along the
#     spiral it unrolls to) — not a class label, but a coordinate used
#     here mainly for coloring points to visualize the roll's structure.
#   - noise=0.05: adds a small amount of random jitter to each point,
#     so the surface isn't perfectly smooth (more realistic, and avoids
#     degenerate/perfectly-flat cases that can trip up some algorithms).

# Make it thinner
X[:, 1] *= .5
# Scale down the second coordinate (column index 1) by half, compressing
# the roll along that one axis — purely a cosmetic adjustment to make
# the shape easier to see clearly in the 3D plot (less "puffy").

# Create a 3D scatter plot
fig = px.scatter_3d(None, x=X[:,0], y=X[:,1], z=X[:,2], color=y,)
# Build an interactive 3D scatter plot using Plotly Express:
#   - None: no DataFrame source; the data is passed directly via x=, y=, z=.
#   - x=X[:,0], y=X[:,1], z=X[:,2]: the 3 coordinate columns of the
#     Swiss roll points.
#   - color=y: colors each point according to its position along the
#     unrolled surface (the y values from make_swiss_roll) — this is
#     what makes the "roll" structure visually obvious as a smooth
#     color gradient spiraling through 3D space, rather than looking
#     like a random blob of same-colored points.

# Update chart looks
fig.update_layout(#title_text="Swiss Roll",
                  showlegend=False,
                  scene_camera=dict(up=dict(x=0, y=0, z=1),
                                        center=dict(x=0, y=0, z=-0.1),
                                        eye=dict(x=1.25, y=1.5, z=1)),
                                        margin=dict(l=0, r=0, b=0, t=0),
                  scene = dict(xaxis=dict(backgroundcolor='white',
                                          color='black',
                                          gridcolor='#f0f0f0',
                                          title_font=dict(size=10),
                                          tickfont=dict(size=10),
                                         ),
                               yaxis=dict(backgroundcolor='white',
                                          color='black',
                                          gridcolor='#f0f0f0',
                                          title_font=dict(size=10),
                                          tickfont=dict(size=10),
                                          ),
                               zaxis=dict(backgroundcolor='lightgrey',
                                          color='black',
                                          gridcolor='#f0f0f0',
                                          title_font=dict(size=10),
                                          tickfont=dict(size=10),
                                         )))
# Customize the figure's visual styling, purely aesthetic (doesn't
# change the underlying data or algorithm):
#   - title_text is commented out, so no title is shown.
#   - showlegend=False: hides the color legend/colorbar side panel.
#   - scene_camera: sets the INITIAL 3D viewing angle when the plot
#     first renders — 'up' defines which direction is "up" on screen,
#     'center' shifts the focal point slightly, 'eye' sets the
#     camera's position relative to the data, controlling the initial
#     tilt/rotation so the roll shape is visible right away (the user
#     can still freely rotate it afterward, since Plotly 3D plots are
#     interactive).
#   - margin=dict(l=0, r=0, b=0, t=0): removes extra whitespace padding
#     around the plot (left/right/bottom/top margins set to 0).
#   - scene=dict(xaxis=..., yaxis=..., zaxis=...): styles each of the
#     3 axes individually — background color, text/tick color, grid
#     line color, and font sizes for axis titles and tick labels. The
#     z-axis gets a slightly different (light grey) background than
#     x/y (white), purely a visual choice to help distinguish the
#     "floor" plane from the vertical axis.

# Update marker size
fig.update_traces(marker=dict(size=3,
                              line=dict(color='black', width=0.1)))
# Adjust how each data point marker is drawn:
#   - size=3: small dot size, appropriate given there are 2000 points
#     (larger markers would overlap and obscure the roll's shape).
#   - line=dict(color='black', width=0.1): gives each marker a very
#     thin black outline, which helps individual points stand out
#     slightly against similarly-colored neighboring points.

fig.update(layout_coloraxis_showscale=False)
# Hide the color scale/colorbar legend that would otherwise show the
# mapping from color to the y (unrolled position) values — redundant
# here since showlegend=False was already set above, but this
# specifically targets the coloraxis scale bar as a belt-and-suspenders
# way of ensuring no color legend appears.

fig.show()
# Render and display the interactive 3D scatter plot in the notebook
# output. The user can click-and-drag to rotate it, scroll to zoom,
# and hover over points to see their values — useful here specifically
# to visually confirm the Swiss roll's spiral structure before running
# MDS (or other manifold learning methods) on it, and later to compare
# against the flattened 2D result.

In [6]:
### Step 1 - Configure MDS function, note we use default hyperparameter values for this example
model2d = MDS(n_components=2,
            n_init=1,
            max_iter=300,
            eps=0.001,
            verbose=1,
            random_state=0,
            dissimilarity='euclidean')
# Create (but don't yet run) an MDS instance, configured to reduce the
# data to 2 dimensions. Key parameters:
#   - n_components=2: the target dimensionality of the output —
#     matches the earlier "R3 -> R2" goal for the Swiss roll.
#   - n_init=1: how many times to run the optimization from different
#     random starting layouts, keeping the best (lowest-stress) result.
#     MDS's optimization can land in different local minima depending
#     on its random initialization, so normally you'd run this several
#     times (e.g. n_init=4) and keep the best; here it's set to just 1
#     run for speed/simplicity in this example.
#   - max_iter=300: the maximum number of iterations the optimizer is
#     allowed to take while minimizing stress, per initialization.
#   - eps=0.001: a convergence tolerance — if the stress improves by
#     less than this amount between iterations, the optimizer stops
#     early rather than running all max_iter iterations.
#   - verbose=1: print progress messages during fitting (iteration
#     number, current stress) so you can watch the optimization happen.
#   - random_state=0: fixes the random seed used for the initial
#     layout, making the result reproducible run-to-run.
#   - dissimilarity='euclidean': tells MDS to compute pairwise
#     dissimilarities (the d_ij values from the stress formula) as
#     ordinary Euclidean distance directly on the raw input X — as
#     opposed to 'precomputed', which would let you pass in your own
#     distance matrix instead of raw coordinates.

### Step 2 - Fit the data and transform it, so we have 2 dimensions instead of 3
X_transformed = model2d.fit_transform(X)
# Run the actual MDS computation on X (the 3D Swiss roll points):
#   - internally computes the 2000x2000 pairwise Euclidean distance
#     matrix from the 3D coordinates,
#   - then iteratively searches for 2D coordinates that minimize the
#     stress between those original 3D distances and the new 2D
#     distances,
#   - returns X_transformed: shape (n_samples, 2), the resulting 2D
#     embedding — one (x, y) pair per original 3D point.
# Unlike PCA's fit/transform, there's no separate "learn parameters,
# then apply to new data" split here in the usual sense — MDS solves
# directly for the embedding of the exact points given; it doesn't
# naturally generalize to transform brand-new, unseen points afterward
# the way PCA's fit-then-transform-on-new-data can.

### Step 3 - Print a few stats
print('The new shape of X: ',X_transformed.shape)
# Confirms the reduction worked: expected output (2000, 2) — 2000
# Swiss roll points, now each represented by 2 coordinates instead of 3.

print('No. of Iterations: ', model2d.n_iter_)
# The actual number of optimization iterations used before stopping
# (either hitting max_iter, or converging early per the eps tolerance).
# e.g. on a smaller 500-point test run, this came out to 30.

print('Stress: ', model2d.stress_)
# The final stress value (from the formula given earlier) — the sum of
# squared differences between original and embedded pairwise distances
# at the point the optimizer stopped. Lower is better, but note this
# raw value isn't normalized (it scales with the number of points and
# the units of the original distances), so it's mainly useful for
# comparing between multiple MDS runs on the SAME dataset (e.g.
# different n_init seeds, or 2D vs. 3D target), not as an absolute
# quality score on its own.

# Dissimilarity matrix contains distances between data points in the original high-dimensional space
#print('Dissimilarity Matrix: ', model2d.dissimilarity_matrix_)
# Would print the full n x n matrix of original
# pairwise distances computed in Step 2 — the d_ij values.

# Embedding contains coordinates for data points in the new lower-dimensional space
#print('Embedding: ', model2d.embedding_)
# Would print the same values as X_transformed —
# model2d.embedding_ is an alternate way to access the fitted 2D
# coordinates directly from the model object, without keeping the
# fit_transform return value.

breaking at iteration 53 with stress 3409113.1337949396
The new shape of X:  (2000, 2)
No. of Iterations:  54
Stress:  3409113.1337949396


We can see that the shape of the new array is 2000 by 2, which means that we have successfully reduced it to 2 dimensions. Also, it took the algorithm 54 iterations to reach the lowest Stress level.

In [7]:
# Create a scatter plot
fig = px.scatter(None, x=X_transformed[:,0], y=X_transformed[:,1], opacity=1, color=y)
# Build a 2D scatter plot (not 3D this time, since the data has already
# been reduced) using the MDS output:
#   - None: no DataFrame source, data passed directly via x=, y=.
#   - x=X_transformed[:,0], y=X_transformed[:,1]: the 2 new MDS
#     coordinates for each point — this is the actual "flattened"
#     result being visualized.
#   - opacity=1: fully opaque markers (no transparency).
#   - color=y: colors each point by its original position along the
#     unrolled Swiss roll surface — the SAME y values used to color
#     the 3D plot earlier, so you can visually compare: if MDS worked
#     well, the smooth color gradient/spiral pattern from the 3D plot
#     should still appear as a coherent, unrolled pattern here in 2D
#     (e.g. a smooth color gradient across a flat sheet), rather than
#     looking scrambled or randomly mixed.

# Change chart background color
fig.update_layout(dict(plot_bgcolor = 'white'))
# Set the plot's background (behind the data points) to white —
# a styling choice, distinct from the page/figure background.

# Update axes lines
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgrey',
                 zeroline=True, zerolinewidth=1, zerolinecolor='lightgrey',
                 showline=True, linewidth=1, linecolor='black')

fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgrey',
                 zeroline=True, zerolinewidth=1, zerolinecolor='lightgrey',
                 showline=True, linewidth=1, linecolor='black')
# Style both axes identically:
#   - showgrid=True + gridwidth/gridcolor: draw light grey gridlines
#     across the plot at each tick, to help read approximate values.
#   - zeroline=True + zerolinewidth/zerolinecolor: draw a distinct line
#     at x=0 (and y=0) — here styled the same light grey as the regular
#     gridlines, so it doesn't stand out much differently.
#   - showline=True + linewidth/linecolor: draw a solid black border
#     line along each axis itself (the axis "spine"), for a cleaner,
#     more defined plot frame compared to Plotly's default.
# This is a 2D-plot equivalent of the axis styling applied to the 3D
# scene earlier — same visual polish, adapted for px.scatter's simpler
# 2D axis API (update_xaxes/update_yaxes) instead of the 3D scene=dict(...).

# Set figure title
fig.update_layout(title_text="MDS Transformation")
# Add a visible title above the plot — unlike the 3D plot earlier,
# where the title was commented out, this one is shown.

# Update marker size
fig.update_traces(marker=dict(size=5,
                             line=dict(color='black', width=0.2)))
# Set marker size to 5 (slightly larger than the 3D plot's size=3,
# reasonable since a 2D plot has less visual clutter/overlap than a
# rotatable 3D one) with a thin black outline per point, same purpose
# as before — helping individual points stand out from same-colored
# neighbors.

fig.show()
# Render and display the interactive 2D scatter plot — the final
# result of the MDS pipeline, letting you visually judge whether the
# 2D layout successfully "unrolled" the Swiss roll's spiral structure
# into a flat, smoothly color-graded sheet, or whether it came out
# tangled/distorted.

The results are pretty good since we could preserve the global structure while at the same time not losing the separation observed between points in the original depth dimension.

In [8]:
## Comparison with PCA
from sklearn.decomposition import PCA
# Import PCA again (already used in the earlier notebook) — brought
# back here specifically to run it on the SAME Swiss roll data as a
# direct comparison against MDS's result, both reducing R3 -> R2.

### Make an instance of the PCA class
pca = PCA(n_components=2)
# Create a PCA instance configured to keep 2 components — matching
# MDS's n_components=2, so both methods produce directly comparable
# 2D outputs from the same 3D input.

## Fit the data and transform it, so we have 2 dimensions instead of 3
X_trans_PCA = pca.fit_transform(X)
# Fit PCA on the raw 3D Swiss roll coordinates X and immediately
# project them onto the 2 principal components (fit + transform in one
# call, as seen before) — giving X_trans_PCA, shape (2000, 2).
# Note: unlike the earlier Iris/Melbourne examples, X here is NOT
# standardized first — it's the raw Swiss roll coordinates straight
# from make_swiss_roll (after the y-axis thinning step). That's likely
# fine for this specific comparison, since all 3 Swiss roll coordinates
# are already on a similar synthetic scale (no units like cm vs. m2
# mixing here), unlike the Melbourne housing case where standardization
# was essential.

## PCA Scatter Plot

This cell visualizes the result of applying PCA to the Swiss roll data, using the same 2D scatter plot style as the earlier MDS visualization, so the two can be compared side by side.

- A 2D scatter plot is created from `X_trans_PCA`, the Swiss roll projected onto its first 2 principal components. Points are colored by `y`, the same value used throughout that represents each point's true position along the rolled-up surface — this lets us visually check whether the reduction preserved the roll's underlying structure.
- The plot background is set to white, and both axes are styled with light grey gridlines, a distinct zero line, and solid black axis borders, matching the look of the earlier MDS plot exactly.
- The title is set to "PCA Transformation" to distinguish this chart from the "MDS Transformation" plot shown previously.
- Marker size and outline are set the same way as before, for visual consistency across both comparison plots.

**Purpose:** since this is the same plotting code as the MDS visualization — only the input data and title differ — its role is purely comparative. The two charts, viewed together, are meant to show how differently a **linear** method (PCA) and a **distance-preserving** method (MDS) handle data with an inherently nonlinear, curved structure like the Swiss roll. Because PCA can only project onto flat (linear) directions of maximum variance, it has no way to "account for" the roll's curvature — so the color gradient in this plot is expected to look noticeably more scrambled or overlapping than in the MDS plot, where points that were close together on the actual spiral tend to stay closer together in the 2D result.

In [9]:
# Create a scatter plot
fig = px.scatter(None, x=X_trans_PCA[:,0], y=X_trans_PCA[:,1], opacity=1, color=y)

# Change chart background color
fig.update_layout(dict(plot_bgcolor = 'white'))

# Update axes lines
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgrey',
                 zeroline=True, zerolinewidth=1, zerolinecolor='lightgrey',
                 showline=True, linewidth=1, linecolor='black')

fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgrey',
                 zeroline=True, zerolinewidth=1, zerolinecolor='lightgrey',
                 showline=True, linewidth=1, linecolor='black')

# Set figure title
fig.update_layout(title_text="PCA Transformation")

# Update marker size
fig.update_traces(marker=dict(size=5,
                             line=dict(color='black', width=0.2)))

fig.show()

While it depends on the exact problem we want to solve, MDS seems to perform better in this scenario than PCA (Principal Component Analysis). For comparison, the below graph shows a 2D representation of the same 3D swiss roll after applying PCA transformation.

As you can see, PCA gives us a result that looks like a picture from one side of the swiss roll, failing to preserve depth information from the third dimension.